<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_73_spark_delta_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🔥 **Module 73 — Spark + Delta Lake (the data-engineering foundation)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> 🎉 **Final module.** This is the closer of the 73-module arc.

> Phase 9 · Production Gaps


# Module 73 — Spark + Delta Lake

> The whole course assumes a Pandas-shaped dataframe fits in memory. **Production data doesn't.** A real ML platform reads from terabytes of event logs, joins to billions of user rows, materialises features (M69), and feeds the result to training (M57) and serving (M44, M71). The dominant tool for that since 2010 has been **Apache Spark**. The 2020s sequel — **Delta Lake** — bolts ACID transactions onto raw Parquet files, turning a data lake into a database. Together they are the **lakehouse**: cheap object storage + warehouse-grade SQL.
>
> This is the final module of the course. By the end you can ingest, transform, version, and merge data at any scale.

### What you'll cover
1. Why distributed data processing (the memory wall)
2. **Spark's architecture** — driver, executors, the DAG
3. PySpark DataFrame API + **Spark SQL**
4. **Lazy evaluation** + **Catalyst** + **Tungsten** — why Spark is fast
5. Joins, windows, UDFs, broadcasts — the hot patterns
6. **Delta Lake** — ACID on top of Parquet
7. **Time travel** + **MERGE INTO** + **schema evolution**
8. **Medallion architecture** — Bronze / Silver / Gold
9. Streaming with **Structured Streaming**
10. Spark vs **Polars** / **DuckDB** / **Snowflake** — when to pick what
11. 🎓 **Course wrap-up** — 73 modules, what you can now build


## 1 · The memory wall

| Tool | Comfortable up to | Past that |
|---|---|---|
| Pandas (M2) | ~1 GB / 10 M rows | swap → OOM |
| **Polars** | ~50 GB on one machine (out-of-core) | needs a cluster |
| **DuckDB** | terabytes on one machine, single-process | analytic queries only |
| **Spark** | petabytes across many machines | the answer at scale |
| **Snowflake / BigQuery / Redshift** | petabytes (warehouse) | hosted; pay per query |

Once your data doesn't fit in one box's RAM, you need **distributed compute**. Spark partitions data across N executors, runs each partition in parallel, and shuffles between stages when joins / aggregations cross partitions.

## 2 · Spark architecture in one picture

```
                    ┌─────────── DRIVER ───────────┐
                    │  - your Python / Scala / SQL │
                    │  - builds the logical plan   │
                    │  - schedules stages          │
                    │  - collects results          │
                    └──────────────┬───────────────┘
                                   │ tasks
        ┌──────────────┬───────────┼───────────┬──────────────┐
        ▼              ▼           ▼           ▼              ▼
   ┌─────────┐   ┌─────────┐  ┌─────────┐ ┌─────────┐   ┌─────────┐
   │EXECUTOR │   │EXECUTOR │  │EXECUTOR │ │EXECUTOR │   │EXECUTOR │
   │ part 0  │   │ part 1  │  │ part 2  │ │ part 3  │   │ part 4  │
   │   ...   │   │   ...   │  │   ...   │ │   ...   │   │   ...   │
   └─────────┘   └─────────┘  └─────────┘ └─────────┘   └─────────┘
        ▼              ▼           ▼           ▼              ▼
   ┌────────────── storage layer (S3 / GCS / ABFS / HDFS) ───────┐
   │   Parquet / Delta / ORC / JSON / CSV files in object store    │
   └──────────────────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| **Driver** | runs *your* program; builds the DAG; schedules tasks |
| **Cluster manager** | YARN / Kubernetes / Standalone — allocates executors |
| **Executor** | JVM process running on a worker node; runs N parallel **tasks** |
| **Task** | unit of work over one **partition** of the data |
| **Partition** | a shard of a DataFrame; typically 128 MB |

The cluster manager (k8s / YARN / Databricks) starts executors. The driver hands them tasks. Executors return results. That's the whole model.

## 3 · PySpark in Colab + the DataFrame API

In [ ]:
!pip -q install pyspark==3.5.3 delta-spark==3.2.1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# In production you'd connect to a Databricks / EMR / GKE Dataproc cluster.
# Here we run a local single-machine Spark inside Colab — same API.
spark = (SparkSession.builder
         .appName("course_demo")
         .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
         .config("spark.sql.catalog.spark_catalog",
                 "org.apache.spark.sql.delta.catalog.DeltaCatalog")
         .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.1")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version)

In [ ]:
# Make some toy data
import pandas as pd, random, datetime as dt
random.seed(0)

rows = []
for i in range(50_000):
    rows.append({
        "user_id":  random.randint(1, 1_000),
        "amount":   round(random.uniform(1, 500), 2),
        "country":  random.choice(["US","DE","IN","BR","JP"]),
        "event_ts": dt.datetime(2024,1,1) + dt.timedelta(seconds=random.randint(0, 7_776_000)),
        "is_fraud": int(random.random() < 0.03),
    })
df = spark.createDataFrame(pd.DataFrame(rows))
df.show(5)
df.printSchema()

In [ ]:
# The DataFrame API: looks Pandas-shaped but builds a query plan
agg = (df
        .filter(F.col("amount") > 50)
        .groupBy("country")
        .agg(
            F.count("*").alias("n_events"),
            F.avg("amount").alias("avg_amount"),
            F.sum(F.col("is_fraud")).alias("n_fraud"),
        )
        .orderBy(F.desc("n_events"))
)
agg.show()

In [ ]:
# Same query in Spark SQL — completely interchangeable
df.createOrReplaceTempView("events")
spark.sql('''
    SELECT country,
           COUNT(*)         AS n_events,
           AVG(amount)      AS avg_amount,
           SUM(is_fraud)    AS n_fraud
    FROM   events
    WHERE  amount > 50
    GROUP BY country
    ORDER BY n_events DESC
''').show()

**Note what just happened.** The DataFrame API call and the SQL query produced **identical query plans**. Spark's **Catalyst** optimiser parses both into the same logical plan, then to a physical plan, then to RDD operations on executors. **Same speed, same correctness — pick whichever your team reads more clearly.**

## 4 · Lazy evaluation, Catalyst, Tungsten — why Spark is fast

In [ ]:
# Spark transformations are LAZY. The .show()/.collect()/.write are ACTIONS.
big = (df.filter(F.col("amount") > 100)
         .groupBy("country", "is_fraud")
         .agg(F.count("*").alias("n")))

# Inspecting the plan WITHOUT running:
big.explain(mode="formatted")

**The plan is built up before anything runs.** That lets the **Catalyst optimiser** rewrite it — push filters down to file scans, prune columns you don't need, choose the cheapest join algorithm, etc. **Tungsten** then turns the physical plan into vectorised CPU code that runs on the **off-heap binary format** instead of JVM objects (faster + lower GC).

A practical consequence: **call `.cache()` on a DataFrame you'll reuse** so the optimiser materialises it once instead of replaying the plan.

In [ ]:
big.cache()
big.count(); big.count()    # second call hits the cache — no re-scan

## 5 · The hot patterns

```python
# JOIN — broadcast small dimension table
from pyspark.sql.functions import broadcast
joined = df.join(broadcast(countries_df), "country")

# WINDOW functions — rolling features (M69 in spirit)
from pyspark.sql.window import Window
w = Window.partitionBy("user_id").orderBy("event_ts")
df.withColumn("running_total", F.sum("amount").over(w)).show(5)

# UDF — when you really must drop to Python (slow!)
@F.pandas_udf("double")               # vectorised pandas-UDF — much faster than per-row
def doubled(x: pd.Series) -> pd.Series:
    return x * 2
df.withColumn("dbl", doubled("amount")).show(3)

# PARTITIONING for write performance — Hive-style directories
df.write.partitionBy("country").parquet("/tmp/events_parquet")
```

**Three rules of fast Spark.**
1. **Always favour built-in `F.*` functions over UDFs**; built-ins compile to JVM/Tungsten code.
2. **`broadcast()` small tables (< ~10 MB)** to avoid expensive shuffle joins.
3. **Repartition before wide operations**; `df.coalesce(N)` to shrink before writing.

## 6 · Delta Lake — ACID on top of Parquet

Spark on plain Parquet has one big hole: **concurrent writes corrupt the data**. Two jobs writing to `s3://bkt/events/` will overwrite each other's files. There's no transaction log. **Delta Lake** fixes that by adding a `_delta_log/` folder of JSON commits next to the Parquet files. Every write is a transaction; every read sees a consistent snapshot.

In [ ]:
# Write the same data as a Delta table
df.write.format("delta").mode("overwrite").save("/tmp/events_delta")

# Read it back
events = spark.read.format("delta").load("/tmp/events_delta")
events.count()

**What you got for free vs Parquet.**

| Feature | Plain Parquet | Delta |
|---|---|---|
| ACID on concurrent writes | ❌ | ✅ |
| Schema enforcement | manual | automatic |
| Schema evolution (`mergeSchema`) | manual | one flag |
| `UPDATE` / `DELETE` / `MERGE INTO` | ❌ | ✅ |
| Time travel (read v3 of the table) | ❌ | ✅ |
| `OPTIMIZE` + `Z-ORDER` for fast skipping | ❌ | ✅ |
| `VACUUM` to delete tombstoned files | ❌ | ✅ |

Two near-equivalents you'll meet: **Apache Iceberg** (Snowflake / AWS / Apple) and **Apache Hudi** (Uber). All three solve the same problem with the same shape; choose the one your stack already standardises on.

## 7 · Time travel + MERGE + schema evolution

In [ ]:
# 1. Time travel — read a previous version
events_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/tmp/events_delta")
events_v0.count()

In [ ]:
# 2. MERGE INTO — upserts, the killer Delta feature
from delta.tables import DeltaTable

# Pretend we receive a daily "updates" feed
updates = spark.createDataFrame(pd.DataFrame([
    {"user_id": 42, "amount": 999.99, "country": "US",
     "event_ts": dt.datetime(2024, 6, 1), "is_fraud": 1},
    {"user_id": 18, "amount": 12.50,  "country": "IN",
     "event_ts": dt.datetime(2024, 6, 1), "is_fraud": 0},
]))

tgt = DeltaTable.forPath(spark, "/tmp/events_delta")
(tgt.alias("t")
    .merge(updates.alias("u"),
           "t.user_id = u.user_id AND t.event_ts = u.event_ts")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

In [ ]:
# 3. Schema evolution — auto-add a new column
new_rows = spark.createDataFrame(pd.DataFrame([
    {"user_id": 7, "amount": 200.0, "country": "FR",
     "event_ts": dt.datetime(2024, 7, 1), "is_fraud": 0,
     "device": "iphone"},                  # ← new column
]))
(new_rows.write.format("delta")
        .option("mergeSchema", "true")     # ← the flag that lets new columns through
        .mode("append")
        .save("/tmp/events_delta"))

**`MERGE INTO`** is what turned data lakes into databases. The same syntax handles inserts, updates, deletes, and slowly-changing-dimensions in one statement. Before Delta, doing an idempotent upsert on Parquet meant writing-then-renaming whole partitions by hand.

**Time travel** is *not* a backup mechanism — old files get reaped by `VACUUM` after the configured retention (default 7 days). Use it for debugging, reproducibility, and `CREATE TABLE clone AS @ <version>` for rollback during deploys.

## 8 · Medallion architecture (Bronze / Silver / Gold)

The canonical lakehouse layout:

```
   raw event firehose
        │
        ▼
   ┌────── BRONZE ─────────┐   raw, append-only, exactly as ingested
   │   /lake/bronze/events │   schema = source schema; cheap; never deleted
   └───────────────────────┘
        │  clean / dedupe / type-cast
        ▼
   ┌────── SILVER ─────────┐   cleaned, conformed, joined
   │   /lake/silver/events │   schema = business schema; the "source of truth"
   └───────────────────────┘
        │  aggregate per use case
        ▼
   ┌────── GOLD ──────────┐    BI-ready, denormalised, smaller tables
   │   /lake/gold/...     │    one folder per dashboard / ML feature view
   └──────────────────────┘
        ▼
   serves: BI dashboards · feature store (M69) · ML training data
```

| Layer | Format | Optimised for | Owners |
|---|---|---|---|
| **Bronze** | Delta append-only | ingest throughput; full audit | data-engineering |
| **Silver** | Delta with `MERGE` | cleanliness; one source of truth | DE + analytics |
| **Gold** | Delta, often Z-ordered | query latency for BI / features | analytics / ML |

Originally a Databricks naming convention; now industry-standard. Build new pipelines top-down, **always** through these layers — short-circuiting from Bronze → Gold guarantees pain.

## 9 · Streaming with Structured Streaming

Spark's micro-batch streaming API. **Same DataFrame code** for batch and streaming — change `.read` to `.readStream` and Spark runs the same plan continuously.

```python
events_stream = (spark.readStream
                      .format("delta")
                      .load("/lake/bronze/events"))

fraud_per_min = (events_stream
                  .withWatermark("event_ts", "5 minutes")
                  .groupBy(F.window("event_ts", "1 minute"), "country")
                  .agg(F.sum("is_fraud").alias("frauds")))

query = (fraud_per_min.writeStream
                       .format("delta")
                       .option("checkpointLocation", "/lake/_checkpoints/fraud_per_min")
                       .outputMode("append")
                       .start("/lake/gold/fraud_per_min"))
```

Two concepts you'll need:
- **Watermark** (`withWatermark`) — Spark drops late records older than the watermark to bound state memory.
- **Checkpoint** — durable state for exactly-once semantics. `checkpointLocation` is non-optional in production.

**Alternatives:** **Apache Flink** for true low-latency (millisecond) streaming and complex event-time logic; **Kafka Streams** when you already live inside Kafka.

## 10 · Spark vs Polars vs DuckDB vs Snowflake

| Tool | Sweet spot | Why pick it |
|---|---|---|
| **Spark** | distributed, petabytes, joins across many tables | the workhorse; matches every cloud; Delta + Iceberg + Hudi support |
| **Polars** (M2-era extension) | single-machine, < 100 GB, lazy Rust performance | dramatically faster than Pandas; one machine, one process |
| **DuckDB** | analytic SQL on local files, terabytes, embedded | "SQLite for analytics" — incredible for ad-hoc + dbt |
| **Snowflake** | hosted warehouse, simple SQL, no ops | pay-per-query; great for BI + dbt-driven transforms |
| **BigQuery** | hosted warehouse on GCP | serverless SQL; native GIS; ML SQL extensions |
| **Trino / Presto** | federated SQL across many sources | query S3 + Postgres + Kafka in one statement |
| **Apache Flink** | low-latency streaming (ms) | when Structured Streaming's batch is too slow |

**The 2026 default stack** at most data-platform teams:
- **Delta Lake** (or Iceberg) on **S3** for storage.
- **Spark** for big-batch + ETL.
- **DuckDB / Polars** for fast local + interactive analytics on top of the same Parquet/Delta.
- **dbt** for managed SQL transformations.
- **Snowflake / BigQuery** for the BI consumption layer (or the same Delta tables via Athena / Databricks SQL).

## 11 · 🎓 Course wrap-up — 73 modules

```
M1   – M5     Python · Pandas · NumPy · Visualisation · SQL
M6   – M15    Stats · classical ML · evaluation · feature eng · MLOps
M16  – M24    PyTorch · transformers · diffusion · time-series · DeepSeek-V3
M25  – M27    Pydantic · LangChain · Chroma RAG starter
M28  – M31    FastAPI · serving · vector DBs · advanced RAG
M32  – M35    LangChain · LangGraph · LlamaIndex · DSPy
M36  – M40    Postgres+pgvector · Redis · Ollama · Unsloth · TF / Keras
M41  – M45    CUDA · vector-DB shootout · multi-agent · vLLM · gRPC
M46  – M51    Kubernetes · Helm · Terraform · Ansible · Prometheus · OTel
M52  – M54    Go · Rust · TypeScript
M55  – M63    LLMs from Scratch (Raschka): tokenize → assemble → pretrain → SFT → cls → KV-cache → Llama-3 conversion → DPO → BPE
M64  – M67    MCP · multimodal (VLM + Whisper + TTS) · distributed training · RLHF / GRPO
M68  – M71    workflow orchestration · feature stores · LLM evals · Triton
M72  – M73    computer-use agents · Spark + Delta Lake (this module)
```

You can:
- **Build a model** — pretrain a GPT (M55-M57), align it with DPO/GRPO (M62, M67), convert it to Llama 3 (M61).
- **Serve a model** — vLLM (M44) or Triton (M71) on Kubernetes (M46), behind gRPC (M45), with PagedAttention + KV cache (M60).
- **Ship a product** — TypeScript frontend (M54), MCP tools (M64), Vision + Voice (M65), browser agent (M72).
- **Operate a platform** — Terraform infra (M48), Helm releases (M47), Ansible nodes (M49), Prometheus + Grafana + OTel observability (M50–M51).
- **Manage data** — Spark + Delta lakehouse (this module), feature store (M69), workflow orchestration (M68), evals (M70).

Pick **one** thing from this list and **ship it**. Modules don't keep you in business — products do.

### 🙏 Closing
- ⭐ Star the repo: [`github.com/kader-xai/data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)
- 🌐 Companion site: [`kader-xai.github.io/data-science-roadmap`](https://kader-xai.github.io/data-science-roadmap/)
- 🛠️ Take *one* of the 73 notebooks and turn it into something a stranger pays you for.

That's the whole job.

## ✅ Recap (M73)
- Single-node tools (Pandas) hit a memory wall. **Spark** distributes compute across many machines.
- Architecture: **driver → cluster manager → executors → tasks over partitions**.
- **Lazy evaluation** + **Catalyst** + **Tungsten** = same DataFrame API and SQL both reach the same fast plan.
- **Delta Lake** = ACID + time travel + `MERGE INTO` on top of Parquet — turning lakes into databases.
- **Medallion architecture** (Bronze / Silver / Gold) is the standard lakehouse layout.
- **Structured Streaming** = same code as batch, with `.readStream` + watermarks + checkpoints.
- Use **Spark** for distributed; **Polars / DuckDB** for single-node analytics; **Snowflake / BigQuery** for hosted warehouses; **Flink** for true low-latency streams.

🎉 **End of the course.** Go build.
